In [1]:
# root file producer for the QC-TS
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd
from openpyxl import Workbook
import pytz

Welcome to JupyROOT 6.30/04


In [2]:
file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/FBK/PRE-SERIE_FBK_Vendor_QC-TS_CV.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
sensor = array('i', [0])
row = array('i', [0])
column = array('i', [0])
side = array('i', [0])

V = root.std.vector("float")()
CPAD = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("sensor", sensor, 'sensor/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')
tree.Branch("side", side,'side/I')


V.reserve(1000)
CPAD.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("CPAD", "std::vector<float>", CPAD)

txt_files = glob.glob("/Users/icosivi/Desktop/PRE-SERIE/FBK/export/1x1_CV/*.txt")

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_QC-TS_sensor-col-row.xlsx')

counter = 0

for txt in txt_files:
    #print(txt)
    V.clear()
    #IBACK.clear()

    with open(txt,"r") as t:
        lines_list = t.readlines()
        header_v = lines_list[0]
        
        header_v.strip()
        hv = re.split("\s+", header_v)
        
        #for m in hv[6:]:
        #for m in hv[8:]:    
        #    if not m.isspace():
        #        if m:
        #            #print(m)
        #            V.push_back( abs(float(m)) )

        lines = lines_list[1:]
        
        for line in lines:
            line.strip()
            CPAD.clear()
            V.clear()
            ll = re.split( '\s+', line)
            #print(ll[5])
            event[0] = counter
            wafer[0] = int( ll[0] )
            column[0] = int( ll[1] )
            row[0] = int( ll[2] )
            #print(str(column[0])+"   "+str(row[0]))
            sensor[0] = int(wb[(wb['Row'] == row[0]) & (wb['Column'] == column[0])]['Sensor'].iloc[0])
            #print(sensor[0])
            side[0] = 1
              
            i_len = 0
            #for q in ll[7:]:
            for q in ll[6:]:
                if not q.isspace():
                    if q:
                        CPAD.push_back( abs(float(q)) )
                        i_len+=1
                        #if counter==2:
                            #print(q)          
            #if not any(x == "PIN" for x in sensor_types):
            v_len = 0
            for m in hv[6:]:    
              if not m.isspace():
                if m and v_len<i_len:
                  #print(m)
                  V.push_back( abs(float(m)) )
                  v_len+=1
            tree.Fill()
            counter += 1

tree.Write()
file.Write()
file.Close()

In [ ]:
# producer of the xls to register components on the database for the QC-TS
xl_filename="FBK_QC-TS_PRE-SERIES"
wb = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

batch = int()

file_qa = root.TFile.Open("/Users/icosivi/Desktop/PRE-SERIE/FBK/PRE-SERIE_FBK_Vendor_QC-TS_CV.root")
tree_qa = file_qa.Get("Tree")

for j,event in enumerate(tree_qa):

    batch = 1

    wws["A%i" %(j+2)] = 'PRE_FBK_QC-TS_W'+str(event.wafer)+'_S'+str(event.sensor)        
    wws["B%i" %(j+2)] = 'FBK'
    wws["C%i" %(j+2)] = batch
    wws["D%i" %(j+2)] = event.wafer
    wws["E%i" %(j+2)] = 'QC-TS'
    wws["F%i" %(j+2)] = event.row
    wws["G%i" %(j+2)] = event.column
    wws["H%i" %(j+2)] = event.sensor

save_path = '/Users/icosivi/Desktop/PRE-SERIE/FBK/'
wb.save(save_path+xl_filename+".xlsx")

In [4]:
# producer of the xls to register components on the database for the QC-TS
xl_filename="FBK_QC-TS_PRE-SERIES_not-tested-by-fbk"
wb = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

batch = int()

sensor = [35,42,43,48,49,50,51,52]
wafer = [14,15,16,17,18,19,20,22,23,24,26]

wbb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_QC-TS_sensor-col-row.xlsx')

pippo = 0

for j in wafer:
    for k in sensor:

     batch = 1

     wws["A%i" %(pippo+2)] = 'PRE_FBK_QC-TS_W'+str(j)+'_S'+str(k)        
     wws["B%i" %(pippo+2)] = 'FBK'
     wws["C%i" %(pippo+2)] = batch
     wws["D%i" %(pippo+2)] = j
     wws["E%i" %(pippo+2)] = 'QC-TS'
     wws["F%i" %(pippo+2)] = int(wbb[(wbb['Sensor'] == k) ]['Row'].iloc[0])
     wws["G%i" %(pippo+2)] = int(wbb[(wbb['Sensor'] == k) ]['Column'].iloc[0])
     wws["H%i" %(pippo+2)] = k
     pippo += 1

save_path = '/Users/icosivi/Desktop/PRE-SERIE/FBK/'
wb.save(save_path+xl_filename+".xlsx")

In [ ]:
# producer of the xls to register components on the database for the QC-TS

xl_filename="FBK_QC-TS_PRE-SERIES"
ww = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

for i in range(n_serialn):
    wws["A%i" %(serial_num_counter+1)] = 'PRE'+str(serial_num)
    wws["B%i" %(serial_num_counter+1)] = vendor
    wws["C%i" %(serial_num_counter+1)] = batch
    wws["D%i" %(serial_num_counter+1)] = Wafer
    wws["E%i" %(serial_num_counter+1)] = "QC-TS"
    wws["F%i" %(serial_num_counter+1)] = Row
    wws["G%i" %(serial_num_counter+1)] = Column
    
    sensor_number = int(wb[(wb['Row'] == Row) & (wb['Column'] == Column)]['Sensor'].iloc[0])
    wws["H%i" %(serial_num_counter+1)] = sensor_number

save_path = '/Users/icosivi/Desktop/PRE-SERIE/FBK/'
wb.save(save_path+xl_filename+".xlsx")